## Linear Probes Experiment Runner

This notebook runs the linear probes experiment with your choice of model and options configuration.
The experiment can be executed locally or in a Google Colab runtime environment.

In [ ]:
COLAB_RUNTIME = True # Set to true to set up the experiment for a Google Colab runtime

In [ ]:
from pathlib import Path

def clone_repo(
    repo_url: str,
    repo_root: Path,
    git_ref: str = "main"
) -> None:
    """Clone the repository into the Google Colab VM.
    
    Args:
        repo_url (str): The URL of the repository to clone.
        repo_root (Path): The root directory for installing the repository.
        git_ref (str): The branch or tag to checkout.
        force_clone (bool): If True, force the clone even if the repository already exists.
    """
    %cd {str(os.path.basename(repo_root))}
    if not repo_root.exists():
        !git clone {repo_url} {str(repo_root)}
    else:
        print("Repo already exists at:", repo_root)

    %cd {str(repo_root)}
    !git fetch --all -q
    !git reset --hard origin/{git_ref}

    assert (repo_root / "utility_analysis").exists(), (
        f"Expected utility_analysis/ under {repo_root}. "
    )

if COLAB_RUNTIME:
    REPO_URL = "https://github.com/thowell332/llm-preferences.git"
    REPO_ROOT = Path( "/content/llm_preferences").expanduser()

    clone_repo(REPO_URL, REPO_ROOT)

In [3]:
%pip install --upgrade pip
%pip install -r /content/llm_preferences/requirements.txt "jedi>=0.16" "rich<14"

In [ ]:
import yaml
from pathlib import Path
from huggingface_hub import snapshot_download


def resolve_models_yaml_path() -> Path:
    """Find utility_analysis/models.yaml (Colab: under REPO_ROOT; local: walk up from cwd)."""
    if COLAB_RUNTIME:
        return REPO_ROOT / "utility_analysis" / "models.yaml"
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "utility_analysis" / "models.yaml"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find utility_analysis/models.yaml. Open the notebook from the repo root "
        "or set the working directory so utility_analysis/models.yaml is discoverable."
    )


def hf_repo_id_for_model_key(models_path: Path, model_key: str) -> str:
    cfg = yaml.safe_load(models_path.read_text())
    if model_key not in cfg:
        raise KeyError(f"{model_key!r} not found in {models_path}")
    repo_id = cfg[model_key].get("model_name")
    if not repo_id:
        raise ValueError(
            f"{model_key!r} has no model_name (Hugging Face repo id) in {models_path}; "
            "cannot snapshot_download."
        )
    return repo_id


def download_model(model_key: str, model_dir: Path, models_path: Path) -> str:
    """Download weights for model_key using models.yaml -> model_name as repo_id."""
    repo_id = hf_repo_id_for_model_key(models_path, model_key)
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(model_dir),
        local_dir_use_symlinks=False,
    )
    print(f"Downloaded {repo_id} to {model_dir}")
    return repo_id


MODEL_KEY = "llama-31-8b-instruct"

MODELS_PATH = resolve_models_yaml_path()
HF_REPO_ID = hf_repo_id_for_model_key(MODELS_PATH, MODEL_KEY)

if COLAB_RUNTIME:
    MODEL_DIR = Path(f"/content/models/{MODEL_KEY}").expanduser()
else:
    repo_root = MODELS_PATH.parent.parent
    MODEL_DIR = (repo_root / "models" / MODEL_KEY).resolve()

download_model(MODEL_KEY, MODEL_DIR, MODELS_PATH)

In [23]:
import yaml
from pathlib import Path


def patch_models_yaml(
    models_path: Path,
    model_key: str,
    model_dir: Path,
    repo_id: str,
) -> None:
    """Patch models.yaml so the vllm model points at our downloaded MODEL_DIR.
    
    Args:
        models_path (Path): The path to the models.yaml file.
        model_key (str): The key of the model to patch.
        model_dir (Path): The directory to download the model to.
        repo_id (str): The ID of the repository to download the model from.
    """
    models = yaml.safe_load(models_path.read_text())
    assert model_key in models, f"{model_key} not found in models.yaml"

    models[model_key]["path"] = str(model_dir)
    models[model_key]["model_name"] = repo_id

    models_path.write_text(yaml.safe_dump(models, sort_keys=False))
    print(f"Updated {models_path} for {model_key} -> {model_dir}")


# Requires the download cell (sets MODELS_PATH, MODEL_KEY, MODEL_DIR, HF_REPO_ID).
_patch_path = globals().get("MODELS_PATH")
if _patch_path is None:
    if COLAB_RUNTIME:
        _patch_path = REPO_ROOT / "utility_analysis" / "models.yaml"
    else:
        _patch_path = resolve_models_yaml_path()
else:
    _patch_path = Path(_patch_path)

patch_models_yaml(_patch_path, MODEL_KEY, Path(MODEL_DIR), HF_REPO_ID)

In [12]:
# -----------------------------
# Pilot layer sweep (single role) BEFORE full experiment
# -----------------------------
import numpy as np
import torch
from transformers import AutoConfig
import json
import yaml
import subprocess
from pathlib import Path

# Pilot-specific defaults (avoid stale globals from earlier cells).
MODEL_KEY = globals().get("MODEL_KEY", globals().get("MODEL_KEY_FOR_DOWNLOAD", "llama-31-8b-instruct"))
# Prefer GPU when available. Loading 8B fp16 on CPU needs ~16GiB+ RAM; WSL often OOM-kills mid-shard (no traceback).
PILOT_FORCE_CPU = globals().get("PILOT_FORCE_CPU", False)
# Default True: 8-bit GPU load via bitsandbytes (~8–10 GiB VRAM). CPU-staged fp16 needs ~18+ GiB RAM and often hits exit -9 on WSL.
PILOT_USE_BNB_8BIT = globals().get("PILOT_USE_BNB_8BIT", True)
# Train stage needs enough examples after the split (see run_linear_probes train()). Use 1 for collect-only smoke tests.
PILOT_MAX_EXAMPLES = globals().get("PILOT_MAX_EXAMPLES", 20)
PILOT_OPTIONS_PATH = "../../shared_options/options_custom.json"
PILOT_UTILITIES_PATH = f"../../shared_utilities/options_custom/{MODEL_KEY}/results_utilities_{MODEL_KEY}_helpful_assistant.json"
MAX_NEW_TOKENS_FOR_PARSING = globals().get("MAX_NEW_TOKENS_FOR_PARSING", 2)
SAVE_DIR = globals().get("SAVE_DIR", f"results_linear_probes/{MODEL_KEY}")

RUN_PILOT_SWEEP = True

PILOT_ROLE = "helpful assistant"  # single role for fast iteration
PILOT_TARGET = "utility"           # use "rating" to plot test accuracy, or "utility"
PILOT_POSITION = "gen_first"      # "gen_first" or "prompt_last"
PILOT_PROBE_MODE = "all"
PILOT_TEST_FRACTION = 0.3
PILOT_RIDGE_LAMBDA = 1.0
PILOT_SEED = 42

# Layer selection strategy:
# - "sample": representative depths
# - "all": every layer
PILOT_LAYER_MODE = "sample"
PILOT_NUM_SAMPLED_LAYERS = 10

# Keep pilot outputs separate from full runs (avoid spaces in artifact filenames).
PILOT_SAVE_SUFFIX = f"{MODEL_KEY}_pilot_{PILOT_ROLE}".replace(" ", "_")

# Resolve model path from models.yaml and infer number of layers.
utility_dir = Path("/home/thowell332/tufts/value-driven-behavior/utility_analysis")
models_yaml_path = utility_dir / "models.yaml"
with open(models_yaml_path, "r") as f:
    models_cfg = yaml.safe_load(f)
model_cfg = models_cfg[MODEL_KEY]
model_path = model_cfg.get("path") or model_cfg.get("model_name")
if model_path is None:
    raise ValueError(f"No path/model_name found for {MODEL_KEY} in models.yaml")

cfg = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
num_layers = int(getattr(cfg, "num_hidden_layers"))

if PILOT_LAYER_MODE == "all":
    pilot_layers = list(range(num_layers))
elif PILOT_LAYER_MODE == "sample":
    # Representative sample across depth (includes early/mid/late layers).
    sampled = np.linspace(0, num_layers - 1, num=PILOT_NUM_SAMPLED_LAYERS)
    pilot_layers = sorted(set(int(round(x)) for x in sampled))
else:
    raise ValueError(f"Unknown PILOT_LAYER_MODE: {PILOT_LAYER_MODE}")

PILOT_LAYERS_SPEC = ",".join(str(x) for x in pilot_layers)

print("Pilot role:", PILOT_ROLE)
print("Model layers:", num_layers)
print("Pilot layers:", pilot_layers)
print("Pilot layer spec:", PILOT_LAYERS_SPEC)
print("Pilot options path:", PILOT_OPTIONS_PATH)
print("Pilot utilities path:", PILOT_UTILITIES_PATH)

pilot_options_abs = (utility_dir / "experiments" / "linear_probes" / PILOT_OPTIONS_PATH).resolve()
pilot_utils_abs = (utility_dir / "experiments" / "linear_probes" / PILOT_UTILITIES_PATH).resolve()
print("Pilot options abs:", pilot_options_abs)
print("Pilot utilities abs:", pilot_utils_abs)
assert pilot_options_abs.exists(), f"Missing options file: {pilot_options_abs}"
assert pilot_utils_abs.exists(), f"Missing utilities file: {pilot_utils_abs}"

In [13]:
import os

# Run pilot collect/train and plot score-vs-layer.
if RUN_PILOT_SWEEP:
    pilot_collect_overrides = {
        "model_key": MODEL_KEY,
        "stage": "collect",
        "backend": "hf",
        "save_dir": SAVE_DIR,
        "save_suffix": PILOT_SAVE_SUFFIX,
        "options_path": PILOT_OPTIONS_PATH,
        "utilities_path": PILOT_UTILITIES_PATH,
        "roles": PILOT_ROLE,
        "layers": PILOT_LAYERS_SPEC,
        "max_new_tokens_for_parsing": MAX_NEW_TOKENS_FOR_PARSING,
        "fp16": True,
        "trust_remote_code": True,
        "max_examples": PILOT_MAX_EXAMPLES,
        "cuda_launch_blocking": True,
        "attn_implementation": "eager",
        "max_model_len": 256,
        "force_cpu": PILOT_FORCE_CPU,
        "hf_bnb_8bit": PILOT_USE_BNB_8BIT,
    }

    pilot_collect_cfg = utility_dir / "linear_probes_collect_pilot_override.yaml"
    with open(pilot_collect_cfg, "w") as f:
        yaml.safe_dump(pilot_collect_overrides, f, sort_keys=False)

    print("Pilot collect override YAML:")
    print(yaml.safe_dump(pilot_collect_overrides, sort_keys=False))

    cmd_collect = [
        "python3", "-u", "run_experiments.py",
        "--experiments", "linear_probes_collect",
        "--models", MODEL_KEY,
        "--config", str(pilot_collect_cfg),
        "--overwrite_results",
    ]
    print("Running pilot collect:", " ".join(cmd_collect))
    pilot_env = {**os.environ, "PYTHONFAULTHANDLER": "1"}
    rc = subprocess.run(cmd_collect, cwd=utility_dir, text=True, capture_output=True, env=pilot_env)
    if rc.returncode != 0:
        print("\n--- pilot collect STDOUT ---\n")
        print(rc.stdout)
        print("\n--- pilot collect STDERR ---\n")
        print(rc.stderr)
        raise RuntimeError(f"Pilot collect failed with code {rc.returncode}")

    pilot_train_overrides = {
        "model_key": MODEL_KEY,
        "stage": "train",
        "save_dir": SAVE_DIR,
        "save_suffix": PILOT_SAVE_SUFFIX,
        "position": PILOT_POSITION,
        "target": PILOT_TARGET,
        "probe_mode": PILOT_PROBE_MODE,
        "test_fraction": PILOT_TEST_FRACTION,
        "ridge_lambda": PILOT_RIDGE_LAMBDA,
        "seed": PILOT_SEED,
    }

    pilot_train_cfg = utility_dir / "linear_probes_train_pilot_override.yaml"
    with open(pilot_train_cfg, "w") as f:
        yaml.safe_dump(pilot_train_overrides, f, sort_keys=False)

    cmd_train = [
        "python3", "-u", "run_experiments.py",
        "--experiments", "linear_probes_train",
        "--models", MODEL_KEY,
        "--config", str(pilot_train_cfg),
        "--overwrite_results",
    ]
    print("Running pilot train:", " ".join(cmd_train))
    rc = subprocess.run(cmd_train, cwd=utility_dir, text=True, capture_output=True, env=pilot_env)
    if rc.returncode != 0:
        print("\n--- pilot train STDOUT ---\n")
        print(rc.stdout)
        print("\n--- pilot train STDERR ---\n")
        print(rc.stderr)
        raise RuntimeError(f"Pilot train failed with code {rc.returncode}")

    import matplotlib.pyplot as plt

    pilot_results_path = (
        utility_dir
        / SAVE_DIR
        / f"linear_probes_{PILOT_SAVE_SUFFIX}_probe_results_{PILOT_POSITION}_{PILOT_TARGET}_{PILOT_PROBE_MODE}.json"
    )
    print("Loading pilot results:", pilot_results_path)

    with open(pilot_results_path, "r") as f:
        pilot_results = json.load(f)

    pilot_metrics = pilot_results["metrics_by_layer"]
    layers = sorted(int(k) for k in pilot_metrics.keys())

    # Prefer accuracy when rating-target probes expose it; otherwise use R2.
    metric_name = "accuracy" if "accuracy" in next(iter(pilot_metrics.values())) else "r2"
    vals = [pilot_metrics[str(l)][metric_name] for l in layers]

    best_layer = layers[int(np.argmax(vals))]
    print(f"Pilot best layer by {metric_name}: {best_layer} ({max(vals):.4f})")

    plt.figure(figsize=(9, 4))
    plt.plot(layers, vals, marker="o")
    plt.axvline(best_layer, linestyle="--", alpha=0.4)
    plt.xlabel("Layer")
    plt.ylabel(f"Test {metric_name}")
    plt.title(f"Pilot sweep ({PILOT_TARGET}, {PILOT_POSITION}, role={PILOT_ROLE})")
    plt.grid(alpha=0.2)
    plt.show()

    print("Suggested next step:")
    print("- Use this curve to pick a layer band (e.g., peak +/- 2 layers) for the full run.")
else:
    print("Skipping pilot sweep (set RUN_PILOT_SWEEP=True to enable).")

In [ ]:
# -----------------------------
# User-configurable experiment settings
# -----------------------------
# Keep this equal to MODEL_KEY_FOR_DOWNLOAD (or rerun the download cell with a new key).
MODEL_KEY = globals().get("MODEL_KEY_FOR_DOWNLOAD", "llama-31-8b-instruct")

# Inputs
OPTIONS_PATH = "../../shared_options/options_custom.json"
ROLES_CONFIG_PATH = "../../shared_options/role_sets.yaml"
ROLESET = "refined_default"
# Point this to your precomputed utilities JSON aligned by option id.
UTILITIES_PATH = f"../../shared_utilities/options_custom/{MODEL_KEY}/results_utilities_{MODEL_KEY}_software_engineer.json"

# Activation extraction settings
LAYERS = "all"                  # e.g. "all", "0-31", "10-20"
MAX_NEW_TOKENS_FOR_PARSING = 2   # keep generation short

# Probe training settings
POSITION = "gen_first"          # "gen_first" or "prompt_last"
TARGET = "utility"              # "utility" or "rating"
PROBE_MODE = "all"              # "all", "per_role", "cross_role"
TEST_FRACTION = 0.2
RIDGE_LAMBDA = 1.0
SEED = 42

# Output
SAVE_DIR = f"results_linear_probes/{MODEL_KEY}"
SAVE_SUFFIX = MODEL_KEY

print("Configured model:", MODEL_KEY)
print("Collect output dir:", SAVE_DIR)

In [ ]:
# Build an override config for the collect stage.
collect_overrides = {
    "model_key": MODEL_KEY,
    "stage": "collect",
    "backend": "vllm",
    "save_dir": SAVE_DIR,
    "save_suffix": SAVE_SUFFIX,
    "options_path": OPTIONS_PATH,
    "utilities_path": UTILITIES_PATH,
    "roleset": ROLESET,
    "roles_config_path": ROLES_CONFIG_PATH,
    "layers": LAYERS,
    "max_new_tokens_for_parsing": MAX_NEW_TOKENS_FOR_PARSING,
    "fp16": True,
    "trust_remote_code": True,
}

collect_cfg_path = utility_dir / "linear_probes_collect_override.yaml"
with open(collect_cfg_path, "w") as f:
    yaml.safe_dump(collect_overrides, f, sort_keys=False)

print("Wrote:", collect_cfg_path)
print(yaml.safe_dump(collect_overrides, sort_keys=False))

In [ ]:
# Run activation/rating collection.
cmd = [
    "python3",
    "-u",
    "run_experiments.py",
    "--experiments", "linear_probes_collect",
    "--models", MODEL_KEY,
    "--config", str(collect_cfg_path),
    "--overwrite_results",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=utility_dir, text=True, capture_output=True)
if proc.returncode != 0:
    print("\n--- collect STDOUT ---\n")
    print(proc.stdout)
    print("\n--- collect STDERR ---\n")
    print(proc.stderr)
    raise RuntimeError(f"Collect stage failed with code {proc.returncode}")

In [ ]:
# Build train override config and run probe training.
train_overrides = {
    "model_key": MODEL_KEY,
    "stage": "train",
    "save_dir": SAVE_DIR,
    "save_suffix": SAVE_SUFFIX,
    "position": POSITION,
    "target": TARGET,
    "probe_mode": PROBE_MODE,
    "test_fraction": TEST_FRACTION,
    "ridge_lambda": RIDGE_LAMBDA,
    "seed": SEED,
}

train_cfg_path = utility_dir / "linear_probes_train_override.yaml"
with open(train_cfg_path, "w") as f:
    yaml.safe_dump(train_overrides, f, sort_keys=False)

print("Wrote:", train_cfg_path)
print(yaml.safe_dump(train_overrides, sort_keys=False))

cmd = [
    "python3",
    "-u",
    "run_experiments.py",
    "--experiments", "linear_probes_train",
    "--models", MODEL_KEY,
    "--config", str(train_cfg_path),
    "--overwrite_results",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=utility_dir, text=True, capture_output=True)
if proc.returncode != 0:
    print("\n--- train STDOUT ---\n")
    print(proc.stdout)
    print("\n--- train STDERR ---\n")
    print(proc.stderr)
    raise RuntimeError(f"Train stage failed with code {proc.returncode}")

In [ ]:
# Quick results load + layer sweep plot.
import matplotlib.pyplot as plt

results_path = (
    utility_dir
    / SAVE_DIR
    / f"linear_probes_{SAVE_SUFFIX}_probe_results_{POSITION}_{TARGET}_{PROBE_MODE}.json"
)
print("Loading:", results_path)

with open(results_path, "r") as f:
    results = json.load(f)

if "metrics_by_layer" in results:
    metrics = results["metrics_by_layer"]
else:
    # For per_role or cross_role modes, aggregate by averaging over splits.
    bucket = []
    if "by_role" in results:
        bucket = [v["metrics_by_layer"] for v in results["by_role"].values()]
    elif "leave_one_role_out" in results:
        bucket = [v["metrics_by_layer"] for v in results["leave_one_role_out"].values()]

    metrics = {}
    for d in bucket:
        for layer, m in d.items():
            metrics.setdefault(layer, {"r2": [], "mse": [], "spearman": []})
            metrics[layer]["r2"].append(m["r2"])
            metrics[layer]["mse"].append(m["mse"])
            metrics[layer]["spearman"].append(m["spearman"])
    metrics = {
        str(k): {
            "r2": float(sum(v["r2"]) / len(v["r2"])),
            "mse": float(sum(v["mse"]) / len(v["mse"])),
            "spearman": float(sum(v["spearman"]) / len(v["spearman"])),
        }
        for k, v in metrics.items()
    }

layers = sorted(int(k) for k in metrics.keys())
r2_vals = [metrics[str(l)]["r2"] for l in layers]
s_vals = [metrics[str(l)]["spearman"] for l in layers]

print("Top 5 layers by R2:")
for l in sorted(layers, key=lambda x: metrics[str(x)]["r2"], reverse=True)[:5]:
    print(f"layer={l:>3}  r2={metrics[str(l)]['r2']:.4f}  spearman={metrics[str(l)]['spearman']:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(layers, r2_vals, label="R2")
plt.plot(layers, s_vals, label="Spearman")
plt.xlabel("Layer")
plt.ylabel("Score")
plt.title(f"Linear Probe Layer Sweep ({POSITION}, target={TARGET}, mode={PROBE_MODE})")
plt.legend()
plt.grid(alpha=0.2)
plt.show()